In [ ]:
import scanpy as sc
import os
import pandas as pd

# --- 1. Configuration ---
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_OUTPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/"
OUTPUT_FILE = os.path.join(MSI_OUTPUT_FOLDER, "mz_alignment_by_animal.csv")

MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]

# Dictionary to store lists of m/z values
mz_data = {}

# --- 2. Extract m/z Names ---
for file_name in MSI_SAMPLE_FILES:
    path = os.path.join(MSI_INPUT_FOLDER, file_name)
    
    # Extract short ID (e.g., YC_1) for the column header
    parts = file_name.split('_')
    sample_id = f"{parts[1]}_{parts[2]}".upper()
    
    print(f"Reading features from {sample_id}...")
    
    # Read only metadata (var) to be extremely fast
    adata = sc.read_h5ad(path, backed='r')
    
    # Store the m/z values as a list under the animal name
    mz_data[sample_id] = adata.var_names.tolist()

# --- 3. Create DataFrame and Save ---
# Note: This handles cases where animals might have different numbers of features 
# by filling empty slots with NaN
df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in mz_data.items()]))

# Optional: Add a row index named 'Feature_Index'
df.index.name = "MZ_Index"

if not os.path.exists(MSI_OUTPUT_FOLDER):
    os.makedirs(MSI_OUTPUT_FOLDER)

df.to_csv(OUTPUT_FILE)

print("\n" + "="*40)
print(f"SUCCESS: Matrix saved to: {OUTPUT_FILE}")
print(f"Columns (Animals): {df.shape[1]}")
print(f"Rows (m/z Peaks):  {df.shape[0]}")
print("="*40)

In [20]:
import scanpy as sc
import os
import pandas as pd

# --- 1. Configuration ---
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_OUTPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"
ALIGNMENT_CSV = "/home/ajarrah/PhD_Thesis/chapter_4/mz_alignment_by_animal.csv"

if not os.path.exists(MSI_OUTPUT_FOLDER):
    os.makedirs(MSI_OUTPUT_FOLDER)

sample_to_file = {
    "YC_1": "halfbrain_yc_1_filtered_common.h5ad", "YC_2": "halfbrain_yc_2_filtered_common.h5ad",
    "YC_3": "halfbrain_yc_3_filtered_common.h5ad", "YC_4": "halfbrain_yc_4_filtered_common.h5ad",
    "YAD_1": "halfbrain_yad_1_filtered_common.h5ad", "YAD_2": "halfbrain_yad_2_filtered_common.h5ad",
    "YAD_3": "halfbrain_yad_3_filtered_common.h5ad", "YAD_4": "halfbrain_yad_4_filtered_common.h5ad",
    "AC_1": "halfbrain_ac_1_filtered_common.h5ad", "AC_2": "halfbrain_ac_2_filtered_common.h5ad",
    "AC_3": "halfbrain_ac_3_filtered_common.h5ad", "AC_4": "halfbrain_ac_4_filtered_common.h5ad",
    "AAD_1": "halfbrain_aad_1_filtered_common.h5ad", "AAD_2": "halfbrain_aad_2_filtered_common.h5ad",
    "AAD_3": "halfbrain_aad_3_filtered_common.h5ad", "AAD_4": "halfbrain_aad_4_filtered_common.h5ad"
}

# --- 2. Load the Alignment CSV ---
df_align = pd.read_csv(ALIGNMENT_CSV)

# Reference values from YAD_2
reference_column = "YAD_2"
# We keep these as floats for the mz column and strings for the var_names
yad2_values_float = df_align[reference_column].astype(float).tolist()
yad2_names_str = df_align[reference_column].astype(str).tolist()

# --- 3. Process Each File ---
for animal_id, file_name in sample_to_file.items():
    input_path = os.path.join(MSI_INPUT_FOLDER, file_name)
    
    if not os.path.exists(input_path):
        print(f"Skipping {animal_id}: File not found.")
        continue

    print(f"Syncing {animal_id} (Index & m/z column) to {reference_column}...")
    
    # Read animal data
    adata = sc.read_h5ad(input_path)
    
    # Create mapping for the Index (strings)
    current_animal_mzs_str = df_align[animal_id].astype(str).tolist()
    rename_map = dict(zip(current_animal_mzs_str, yad2_names_str))
    
    # 1. Update var_names (Index)
    adata.var_names = [rename_map.get(str(name), str(name)) for name in adata.var_names]
    
    # 2. Update the numerical 'mz' column (if it exists) 
    # if your column is named 'mzs', 'mz_values', etc., update here:
    if 'mzs' in adata.var.columns:
        adata.var['mzs'] = yad2_values_float

    # Save as a new file
    output_path = os.path.join(MSI_OUTPUT_FOLDER, f"{file_name[:-5]}_synced.h5ad")
    adata.write(output_path)
    print(f"Saved: {os.path.basename(output_path)}")

print("\n" + "="*50)
print(f"DONE: Index and m/z columns synced to {reference_column}.")
print("="*50)

Syncing YC_1 (Index & m/z column) to YAD_2...
Saved: halfbrain_yc_1_filtered_common_synced.h5ad
Syncing YC_2 (Index & m/z column) to YAD_2...
Saved: halfbrain_yc_2_filtered_common_synced.h5ad
Syncing YC_3 (Index & m/z column) to YAD_2...
Saved: halfbrain_yc_3_filtered_common_synced.h5ad
Syncing YC_4 (Index & m/z column) to YAD_2...
Saved: halfbrain_yc_4_filtered_common_synced.h5ad
Syncing YAD_1 (Index & m/z column) to YAD_2...
Saved: halfbrain_yad_1_filtered_common_synced.h5ad
Syncing YAD_2 (Index & m/z column) to YAD_2...
Saved: halfbrain_yad_2_filtered_common_synced.h5ad
Syncing YAD_3 (Index & m/z column) to YAD_2...
Saved: halfbrain_yad_3_filtered_common_synced.h5ad
Syncing YAD_4 (Index & m/z column) to YAD_2...
Saved: halfbrain_yad_4_filtered_common_synced.h5ad
Syncing AC_1 (Index & m/z column) to YAD_2...
Saved: halfbrain_ac_1_filtered_common_synced.h5ad
Syncing AC_2 (Index & m/z column) to YAD_2...
Saved: halfbrain_ac_2_filtered_common_synced.h5ad
Syncing AC_3 (Index & m/z column

In [17]:
"Results tester"
import scanpy as sc
import os
import pandas as pd

# --- 1. Configuration ---
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/"
MSI_OUTPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/"
OUTPUT_FILE = os.path.join(MSI_OUTPUT_FOLDER, "mz_alignment_by_animal__synced.csv")

MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common_synced.h5ad", "halfbrain_yc_2_filtered_common_synced.h5ad",
    "halfbrain_yc_3_filtered_common_synced.h5ad", "halfbrain_yc_4_filtered_common_synced.h5ad",
    "halfbrain_yad_1_filtered_common_synced.h5ad", "halfbrain_yad_2_filtered_common_synced.h5ad",
    "halfbrain_yad_3_filtered_common_synced.h5ad", "halfbrain_yad_4_filtered_common_synced.h5ad",
    "halfbrain_ac_1_filtered_common_synced.h5ad", "halfbrain_ac_2_filtered_common_synced.h5ad",
    "halfbrain_ac_3_filtered_common_synced.h5ad", "halfbrain_ac_4_filtered_common_synced.h5ad",
    "halfbrain_aad_1_filtered_common_synced.h5ad", "halfbrain_aad_2_filtered_common_synced.h5ad",
    "halfbrain_aad_3_filtered_common_synced.h5ad", "halfbrain_aad_4_filtered_common_synced.h5ad"
]

# Dictionary to store lists of m/z values
mz_data = {}

# --- 2. Extract m/z Names ---
for file_name in MSI_SAMPLE_FILES:
    path = os.path.join(MSI_INPUT_FOLDER, file_name)
    
    # Extract short ID (e.g., YC_1) for the column header
    parts = file_name.split('_')
    sample_id = f"{parts[1]}_{parts[2]}".upper()
    
    print(f"Reading features from {sample_id}...")
    
    # Read only metadata (var) to be extremely fast
    adata = sc.read_h5ad(path, backed='r')
    
    # Store the m/z values as a list under the animal name
    mz_data[sample_id] = adata.var_names.tolist()

# --- 3. Create DataFrame and Save ---
# Note: This handles cases where animals might have different numbers of features 
# by filling empty slots with NaN
df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in mz_data.items()]))

# Optional: Add a row index named 'Feature_Index'
df.index.name = "MZ_Index"

if not os.path.exists(MSI_OUTPUT_FOLDER):
    os.makedirs(MSI_OUTPUT_FOLDER)

df.to_csv(OUTPUT_FILE)

print("\n" + "="*40)
print(f"SUCCESS: Matrix saved to: {OUTPUT_FILE}")
print(f"Columns (Animals): {df.shape[1]}")
print(f"Rows (m/z Peaks):  {df.shape[0]}")
print("="*40)

Reading features from YC_1...
Reading features from YC_2...
Reading features from YC_3...
Reading features from YC_4...
Reading features from YAD_1...
Reading features from YAD_2...
Reading features from YAD_3...
Reading features from YAD_4...
Reading features from AC_1...
Reading features from AC_2...
Reading features from AC_3...
Reading features from AC_4...
Reading features from AAD_1...
Reading features from AAD_2...
Reading features from AAD_3...
Reading features from AAD_4...

SUCCESS: Matrix saved to: /home/ajarrah/PhD_Thesis/chapter_4/mz_alignment_by_animal__synced.csv
Columns (Animals): 16
Rows (m/z Peaks):  528


In [21]:
adata = sc.read_h5ad("/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common_synced/halfbrain_yc_1_filtered_common_synced.h5ad")

In [22]:
adata.var

,mzs,mzs_original
326.1991,326.1991,326.1907
337.1936,337.1936,337.1835
339.2312,339.2312,339.2211
340.2334,340.2334,340.2250
341.2454,341.2454,341.2370
...,...,...
1001.6989,1001.6989,1001.6712
1002.7011,1002.7011,1002.6784
1003.7093,1003.7093,1003.6866
1017.6721,1017.6721,1017.6436
